# 🧬 Módulo 4: Modelado de Proteínas y Docking Molecular
## Actividad 4.8: Cribado Virtual (Virtual Screening)

<div align="center">
  
**Universidad de Caldas - Departamento de Química**  
*Introducción a la Química Computacional (173G7G)*  
**Profesor:** José Mauricio Rodas Rodríguez

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maurorodas/Quimica_computacional_173G7G/blob/main/modulo_04_modelado_proteinas_docking/08_virtual_screening.ipynb)

</div>

---

## 🎯 Objetivos de Aprendizaje

Al finalizar esta actividad, serás capaz de:
- Realizar docking de alto rendimiento
- Preparar librerías de compuestos
- Aplicar filtros ADME y regla de Lipinski
- Analizar resultados estadísticamente
- Priorizar candidatos para pruebas experimentales
- Automatizar flujos de trabajo de cribado

---

## 📚 Introducción

El cribado virtual permite evaluar miles de compuestos computacionalmente para identificar candidatos prometedores para desarrollo de fármacos.

---

In [ ]:
# Instalación de dependencias
!pip install rdkit-pypi pandas numpy matplotlib requests 2>/dev/null || \
  pip install rdkit pandas numpy matplotlib requests
!pip install py3Dmol
print("✓ Dependencias instaladas")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import requests
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

try:
    from rdkit import Chem
    from rdkit.Chem import AllChem, Descriptors, rdMolDescriptors, FilterCatalog
    from rdkit.Chem.FilterCatalog import FilterCatalogParams
    print("✓ RDKit disponible")
    RDKIT_OK = True
except ImportError:
    print("⚠️  RDKit no disponible. Instala: conda install -c conda-forge rdkit")
    RDKIT_OK = False

print("✓ Importaciones completadas")

## 1. Introducción al Cribado Virtual

El **cribado virtual** (*virtual screening*, VS) consiste en evaluar *in silico* grandes colecciones de compuestos para identificar los más prometedores como candidatos a fármacos o sondas biológicas.

### Flujo de trabajo general

```
Biblioteca de compuestos
        │
   ▼ Filtros de calidad
   Eliminar compuestos inestables,
   reactivos o PAINs
        │
   ▼ Filtros ADME (Regla de Lipinski)
   MW < 500, LogP < 5, HBD ≤ 5, HBA ≤ 10
        │
   ▼ Docking molecular (alta capacidad)
   Generar y puntuar poses de unión
        │
   ▼ Análisis de resultados
   Clustering, análisis de similitud,
   curvas ROC/EF
        │
   Candidatos prioritarios
```

### Tipos de cribado virtual

| Método | Base | Velocidad | Exactitud |
|--------|------|-----------|-----------|
| **Filtros fisicoquímicos** | Propiedades | Muy rápida | Baja |
| **QSAR** | Ecuaciones matemáticas | Rápida | Media |
| **Farmacóforo** | Patrón 3D | Media | Media-alta |
| **Docking** | Estructura proteína | Lenta | Alta |
| **Machine Learning** | Datos históricos | Variable | Alta |

## 2. Preparación de la Biblioteca de Compuestos

Usaremos como objetivo la proteasa principal del SARS-CoV-2 (**Mpro**, PDB: **6LU7**), un blanco terapéutico activamente investigado. La biblioteca incluye inhibidores conocidos y compuestos inactivos para el análisis.

In [ ]:
# Biblioteca de compuestos: inhibidores de Mpro y decoys
BIBLIOTECA_SMILES = {
    # Inhibidores conocidos de Mpro (activos)
    "N3 (Crystal ligand)":    "O=C(NC(CC1CCCCC1)C(=O)N2CCOCC2)c1cc(-c2ccccc2)no1",
    "Lopinavir":              "CC(C)CC(NC(=O)c1nc2ccc(Cl)cc2n1C)C(=O)N1CC(=O)NC(Cc2ccccc2Cc2ccccc2)C1",
    "Ritonavir":              "CC(C)c1nc(CN(C)C(=O)N(C)c2nc3ccc(Cl)cc3n2)cs1",
    "Boceprevir":             "O=C(NC(CC(=O)N1CCCC1=O)C(=O)C1CC1)NC1(CC1)C(=O)NC(C(F)(F)F)C1CC1",
    "GC376":                  "O=C(N1CCCC1=O)CC(NC(=O)Cc1ccccc1)C(=O)NCC(=O)c1ccc(OCC)cc1",
    "Nirmatrelvir (Paxlovid)":"CC1(C2CC2NC(=O)C(NC(=O)NC2CCCCC2)C2CC2)C(=O)N1CC(F)(F)F",
    # Fármacos antivirales sin actividad conocida en Mpro (inactivos/controles)
    "Oseltamivir":            "CCOC(=O)C1=C[C@@H](OC(CC)CC)[C@@H](NC(C)=O)[C@@H](N)C1",
    "Aciclovir":              "Nc1nc2c(ncn2COCCO)c(=O)[nH]1",
    "Lamivudine":             "Nc1ccn(C2CS[C@@H](CO)O2)c(=O)n1",
    # Compuestos tipo droga (random ADME-filter passing)
    "Ibuprofen":              "CC(C)Cc1ccc(C(C)C(=O)O)cc1",
    "Aspirin":                "CC(=O)Oc1ccccc1C(=O)O",
    "Caffeine":               "Cn1c(=O)c2c(ncn2C)n(C)c1=O",
    "Metformin":              "CN(C)C(=N)NC(=N)N",
    "Chloroquine":            "CCN(CC)CCCC(C)Nc1ccnc2cc(Cl)ccc12",
}

print(f"Biblioteca generada: {len(BIBLIOTECA_SMILES)} compuestos")
for nombre, smi in BIBLIOTECA_SMILES.items():
    print(f"  • {nombre}: {smi[:50]}...")

## 3. Filtros ADME y Regla de Lipinski

La **Regla de los Cinco de Lipinski** predice la biodisponibilidad oral de un compuesto:

| Propiedad | Símbolo | Umbral |
|-----------|---------|--------|
| Peso molecular | MW | ≤ 500 Da |
| LogP | cLogP | ≤ 5 |
| Donadores de enlace H | HBD | ≤ 5 |
| Aceptores de enlace H | HBA | ≤ 10 |
| Rotatable bonds | RB | ≤ 10 |

Además aplicaremos filtros **PAINS** (Pan-Assay INterference compoundS) que eliminan compuestos que dan falsos positivos en ensayos biológicos.

In [ ]:
def calcular_descriptores(nombre, smiles):
    """Calcula descriptores fisicoquímicos de un compuesto."""
    if not RDKIT_OK:
        return None
    
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    
    mw   = Descriptors.MolWt(mol)
    logp = Descriptors.MolLogP(mol)
    hbd  = rdMolDescriptors.CalcNumHBD(mol)
    hba  = rdMolDescriptors.CalcNumHBA(mol)
    rb   = rdMolDescriptors.CalcNumRotatableBonds(mol)
    tpsa = Descriptors.TPSA(mol)
    n_ar = rdMolDescriptors.CalcNumAromaticRings(mol)
    
    # Regla de Lipinski
    lipinski = (mw <= 500 and logp <= 5 and hbd <= 5 and hba <= 10)
    
    # Filtro adicional Veber (oral bioavailability)
    veber = (rb <= 10 and tpsa <= 140)
    
    # Filtro PAINS
    pains = False
    try:
        params = FilterCatalogParams()
        params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
        catalog = FilterCatalog.FilterCatalog(params)
        pains = catalog.HasMatch(mol)
    except Exception:
        pass
    
    return {
        'Nombre': nombre,
        'MW (Da)': round(mw, 1),
        'LogP': round(logp, 2),
        'HBD': hbd,
        'HBA': hba,
        'RB': rb,
        'TPSA (Å²)': round(tpsa, 1),
        'Arom. Rings': n_ar,
        'Lipinski': '✓' if lipinski else '✗',
        'Veber': '✓' if veber else '✗',
        'PAINS': '✗ FALLA' if pains else '✓ OK',
        'Pasa Filtros': lipinski and veber and not pains,
    }

# Calcular descriptores para toda la biblioteca
datos = []
for nombre, smiles in BIBLIOTECA_SMILES.items():
    d = calcular_descriptores(nombre, smiles)
    if d:
        datos.append(d)

df = pd.DataFrame(datos)
print(f"\n📊 Biblioteca procesada: {len(df)} compuestos")
print(f"   ✓ Pasan todos los filtros: {df['Pasa Filtros'].sum()}")
print(f"   ✗ Fallan algún filtro:     {(~df['Pasa Filtros']).sum()}")
df.drop(columns=['Pasa Filtros'])

In [ ]:
def visualizar_espacio_quimico(df):
    """Visualiza el espacio químico de la biblioteca (gráfico de dispersión)."""
    if df is None or df.empty:
        return
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    colores = ['#2ecc71' if p else '#e74c3c' for p in df['Pasa Filtros']]
    
    # MW vs LogP (espacio de Lipinski)
    ax = axes[0]
    ax.scatter(df['LogP'], df['MW (Da)'], c=colores, s=100, 
              edgecolors='white', linewidth=0.5, alpha=0.85, zorder=3)
    ax.axvline(5, color='gray', linestyle='--', alpha=0.6, label='LogP=5')
    ax.axhline(500, color='gray', linestyle=':', alpha=0.6, label='MW=500')
    ax.fill_between([-3, 5], [0, 0], [500, 500], alpha=0.08, color='green',
                   label='Zona Lipinski')
    for _, row in df.iterrows():
        ax.annotate(row['Nombre'].split()[0], (row['LogP'], row['MW (Da)']),
                   fontsize=6, ha='left', va='bottom', alpha=0.7)
    ax.set_xlabel('LogP', fontsize=11)
    ax.set_ylabel('Peso Molecular (Da)', fontsize=11)
    ax.set_title('Espacio de Lipinski', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # TPSA vs RB
    ax = axes[1]
    ax.scatter(df['RB'], df['TPSA (Å²)'], c=colores, s=100,
              edgecolors='white', linewidth=0.5, alpha=0.85, zorder=3)
    ax.axvline(10, color='gray', linestyle='--', alpha=0.6, label='RB=10')
    ax.axhline(140, color='gray', linestyle=':', alpha=0.6, label='TPSA=140')
    ax.fill_between([0, 10], [0, 0], [140, 140], alpha=0.08, color='blue',
                   label='Zona Veber')
    ax.set_xlabel('Rotatable Bonds', fontsize=11)
    ax.set_ylabel('TPSA (Å²)', fontsize=11)
    ax.set_title('Filtro de Veber\n(Biodisponibilidad Oral)', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Radar/barras con conteo de filtros
    ax = axes[2]
    propiedades = ['MW (Da)', 'LogP', 'HBD', 'HBA', 'RB', 'TPSA (Å²)']
    limites = [500, 5, 5, 10, 10, 140]
    df_num = df[propiedades].copy()
    mediana = df_num.median()
    fracciones = [mediana[p] / limites[i] for i, p in enumerate(propiedades)]
    barras_col = ['#2ecc71' if f <= 1.0 else '#e74c3c' for f in fracciones]
    ax.barh(propiedades, fracciones, color=barras_col, alpha=0.8, edgecolor='white')
    ax.axvline(1.0, color='red', linestyle='--', linewidth=1.5, label='Límite Lipinski/Veber')
    ax.set_xlabel('Fracción del límite (mediana)', fontsize=10)
    ax.set_title('Perfil ADME Mediano\nde la Biblioteca', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3, axis='x')
    
    from matplotlib.patches import Patch
    leyenda = [Patch(color='#2ecc71', label='Pasa filtros'),
               Patch(color='#e74c3c', label='Falla algún filtro')]
    fig.legend(handles=leyenda, loc='lower center', ncol=2, fontsize=9,
              bbox_to_anchor=(0.5, -0.04))
    
    plt.suptitle('Análisis del Espacio Químico de la Biblioteca', 
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

visualizar_espacio_quimico(df)

## 4. Cribado por Docking (Flujo Automatizado)

Con la biblioteca pre-filtrada, ejecutamos docking contra el sitio activo de la **Mpro de SARS-CoV-2** (PDB: 6LU7). El sitio activo se define por la caja centrada en el ligando co-cristalizado N3.

In [ ]:
def ejecutar_cribado_simulado(biblioteca_smiles, target_name="Mpro SARS-CoV-2"):
    """
    Simula un cribado virtual por docking.
    En producción sustituir por llamadas reales a AutoDock Vina.
    Los scores simulados están basados en datos publicados para Mpro.
    """
    # Scores de afinidad simulados (kcal/mol) basados en literatura
    # Inhibidores de Mpro tienen mejores scores (más negativos)
    scores_literatura = {
        "N3 (Crystal ligand)":    (-8.3, 0.2),
        "Lopinavir":              (-8.1, 0.3),
        "Ritonavir":              (-7.8, 0.3),
        "Boceprevir":             (-7.5, 0.4),
        "GC376":                  (-8.6, 0.2),
        "Nirmatrelvir (Paxlovid)":(-8.9, 0.2),
        "Oseltamivir":            (-5.8, 0.5),
        "Aciclovir":              (-4.2, 0.6),
        "Lamivudine":             (-4.8, 0.5),
        "Ibuprofen":              (-5.1, 0.5),
        "Aspirin":                (-4.5, 0.6),
        "Caffeine":               (-5.0, 0.5),
        "Metformin":              (-3.9, 0.7),
        "Chloroquine":            (-6.1, 0.4),
    }
    
    resultados = []
    for nombre, smiles in biblioteca_smiles.items():
        media, std = scores_literatura.get(nombre, (-5.5, 0.8))
        score = np.random.normal(media, std)
        score = round(score, 2)
        
        resultados.append({
            'Compuesto': nombre,
            'SMILES': smiles,
            'ΔG binding (kcal/mol)': score,
            'Ki estimado (μM)': round(np.exp(score / (0.001987 * 298)) * 1e6, 3),
            'Clase': 'Inhibidor Mpro' if 'ligand' in nombre.lower() or nombre in 
                     ["Lopinavir", "Ritonavir", "Boceprevir", "GC376", 
                      "Nirmatrelvir (Paxlovid)"] else 'Control',
        })
    
    df_res = pd.DataFrame(resultados).sort_values('ΔG binding (kcal/mol)')
    df_res['Ranking'] = range(1, len(df_res) + 1)
    return df_res

# Ejecutar cribado
np.random.seed(42)
df_resultados = ejecutar_cribado_simulado(BIBLIOTECA_SMILES)
print(f"\n🔬 RESULTADOS DEL CRIBADO VIRTUAL — Mpro SARS-CoV-2")
print(f"{'='*65}")
print(df_resultados[['Ranking', 'Compuesto', 'ΔG binding (kcal/mol)', 
                      'Ki estimado (μM)', 'Clase']].to_string(index=False))

## 5. Análisis Estadístico y Métricas de Enriquecimiento

El análisis cuantitativo evalúa qué tan bien el cribado separa activos de inactivos usando:
- **Curva ROC** (Receiver Operating Characteristic)
- **Factor de enriquecimiento** (EF) al 1%, 5%, 10% de la biblioteca
- **BEDROC** (Boltzmann-Enhanced Discrimination ROC)

In [ ]:
def calcular_enriquecimiento(df_res, col_score='ΔG binding (kcal/mol)', 
                              col_clase='Clase', clase_activa='Inhibidor Mpro'):
    """
    Calcula curva ROC, AUC y factores de enriquecimiento.
    """
    df = df_res.copy().sort_values(col_score)  # Menor ΔG = mejor
    n_total = len(df)
    n_activos = (df[col_clase] == clase_activa).sum()
    n_inactivos = n_total - n_activos
    
    # Curva ROC manual
    activos_sorted = (df[col_clase] == clase_activa).values
    tpr = np.concatenate([[0], np.cumsum(activos_sorted) / n_activos])
    fpr = np.concatenate([[0], np.cumsum(~activos_sorted) / n_inactivos])
    auc = np.trapz(tpr, fpr)
    
    # Factores de enriquecimiento
    def ef_porcentaje(p):
        n_seleccionados = max(1, int(n_total * p / 100))
        n_activos_top = activos_sorted[:n_seleccionados].sum()
        ef = (n_activos_top / n_seleccionados) / (n_activos / n_total)
        return round(ef, 2)
    
    ef1  = ef_porcentaje(10)   # 10% de biblioteca
    ef5  = ef_porcentaje(30)   # 30%
    ef10 = ef_porcentaje(50)   # 50%
    
    # Visualización
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Curva ROC
    ax = axes[0]
    ax.plot(fpr, tpr, 'b-', linewidth=2.5, label=f'AUC = {auc:.3f}')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aleatorio (AUC=0.5)')
    ax.fill_between(fpr, tpr, alpha=0.15, color='blue')
    ax.set_xlabel('Tasa de Falsos Positivos', fontsize=11)
    ax.set_ylabel('Tasa de Verdaderos Positivos', fontsize=11)
    ax.set_title('Curva ROC del Cribado', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Distribución de scores por clase
    ax = axes[1]
    activos_scores = df[df[col_clase] == clase_activa][col_score]
    inactivos_scores = df[df[col_clase] != clase_activa][col_score]
    ax.hist(activos_scores, bins=8, alpha=0.7, color='#2ecc71', label='Inhibidores Mpro')
    ax.hist(inactivos_scores, bins=8, alpha=0.7, color='#e74c3c', label='Controles')
    ax.axvline(activos_scores.mean(), color='green', linestyle='--', alpha=0.8, 
              label=f'Media activos: {activos_scores.mean():.1f}')
    ax.axvline(inactivos_scores.mean(), color='red', linestyle='--', alpha=0.8,
              label=f'Media controles: {inactivos_scores.mean():.1f}')
    ax.set_xlabel('ΔG binding (kcal/mol)', fontsize=11)
    ax.set_ylabel('Frecuencia', fontsize=11)
    ax.set_title('Distribución de Scores\npor Clase', fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Curva de enriquecimiento acumulativo
    ax = axes[2]
    fracciones = np.linspace(0, 1, n_total + 1)
    activos_cum = np.concatenate([[0], np.cumsum(activos_sorted)]) / n_activos
    ax.plot(fracciones, activos_cum, 'b-', linewidth=2.5, label='Docking VS')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Aleatorio')
    ax.fill_between(fracciones, activos_cum, fracciones, alpha=0.15, color='blue')
    ax.set_xlabel('Fracción de la biblioteca', fontsize=11)
    ax.set_ylabel('Fracción de activos recuperados', fontsize=11)
    ax.set_title('Curva de Enriquecimiento', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.suptitle('Métricas de Enriquecimiento — Cribado Virtual Mpro', 
                fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 MÉTRICAS DE ENRIQUECIMIENTO:")
    print(f"  AUC-ROC:  {auc:.3f}  ({'Excelente' if auc > 0.8 else 'Bueno' if auc > 0.6 else 'Regular'})")
    print(f"  EF (top 10%): {ef1:.2f}x")
    print(f"  EF (top 30%): {ef5:.2f}x")
    print(f"  EF (top 50%): {ef10:.2f}x")
    
    return auc, ef1, ef5, ef10

auc, ef1, ef5, ef10 = calcular_enriquecimiento(df_resultados)

## 6. Priorización de Hits y Análisis Multicriterio

Un **hit** de cribado virtual debe superar múltiples criterios antes de pasar a validación experimental:

- **Score de docking** ≤ −7 kcal/mol
- **Cumplimiento ADME** (Lipinski + Veber)
- **Sin alertas PAINS**
- **Selectividad**: comparar con blancos off-target
- **Similitud con fármacos aprobados** o scaffolds conocidos

In [ ]:
def priorizar_hits(df_resultados, df_descriptores, umbral_dg=-7.0):
    """
    Combina scores de docking con filtros ADME para priorizar hits.
    Genera un score compuesto y un heatmap de candidatos.
    """
    # Unir tablas
    df_adme = df_descriptores.copy()
    df_adme.rename(columns={'Nombre': 'Compuesto'}, inplace=True)
    df_merged = df_resultados.merge(df_adme, on='Compuesto', how='left')
    
    # Criterios de selección de hits
    hits = df_merged[df_merged['ΔG binding (kcal/mol)'] <= umbral_dg].copy()
    hits_adme = hits[hits['Pasa Filtros'] == True].copy()
    
    # Score compuesto normalizado (0-100)
    score_dg_norm = (hits_adme['ΔG binding (kcal/mol)'] - 
                     hits_adme['ΔG binding (kcal/mol)'].min()) / \
                    (hits_adme['ΔG binding (kcal/mol)'].max() - 
                     hits_adme['ΔG binding (kcal/mol)'].min() + 1e-9)
    score_tpsa_norm = 1 - (hits_adme['TPSA (Å²)'] / 140)
    score_logp_norm = 1 - (hits_adme['LogP'].abs() / 5)
    
    hits_adme['Score Compuesto'] = round(
        (0.6 * (1 - score_dg_norm) + 
         0.2 * score_tpsa_norm.clip(0, 1) + 
         0.2 * score_logp_norm.clip(0, 1)) * 100, 1
    )
    hits_adme = hits_adme.sort_values('Score Compuesto', ascending=False)
    
    print(f"🎯 PRIORIZACIÓN DE HITS (ΔG ≤ {umbral_dg} kcal/mol + ADME)")
    print(f"  Total con buen docking: {len(hits)}")
    print(f"  Pasan también filtros ADME: {len(hits_adme)}")
    
    if len(hits_adme) > 0:
        print(f"\n🏆 TOP CANDIDATOS:")
        cols = ['Compuesto', 'ΔG binding (kcal/mol)', 'Ki estimado (μM)',
                'MW (Da)', 'LogP', 'TPSA (Å²)', 'Score Compuesto']
        print(hits_adme[cols].to_string(index=False))
        
        # Heatmap de propiedades
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # Heatmap de descriptores normalizados
        propiedades = ['MW (Da)', 'LogP', 'HBD', 'HBA', 'TPSA (Å²)', 'RB']
        limites = [500, 5, 5, 10, 140, 10]
        
        mat = []
        labels_y = []
        for _, row in hits_adme.iterrows():
            frac = [min(row[p] / lim, 1.5) for p, lim in zip(propiedades, limites)]
            mat.append(frac)
            labels_y.append(row['Compuesto'].split()[0][:15])
        
        if mat:
            mat = np.array(mat)
            im = axes[0].imshow(mat, cmap='RdYlGn_r', aspect='auto', vmin=0, vmax=1.5)
            axes[0].set_xticks(range(len(propiedades)))
            axes[0].set_xticklabels(propiedades, rotation=45, ha='right', fontsize=9)
            axes[0].set_yticks(range(len(labels_y)))
            axes[0].set_yticklabels(labels_y, fontsize=9)
            axes[0].set_title('Perfil ADME de Candidatos\n(Verde = OK, Rojo = Excede límite)', 
                            fontsize=11, fontweight='bold')
            plt.colorbar(im, ax=axes[0], label='Fracción del límite')
            
            for i in range(mat.shape[0]):
                for j in range(mat.shape[1]):
                    axes[0].text(j, i, f'{mat[i,j]:.2f}', ha='center', va='center',
                               fontsize=7, color='black')
        
        # Bar chart score compuesto
        y_pos = range(len(hits_adme))
        colores_bar = cm.get_cmap('RdYlGn')(
            np.array(hits_adme['Score Compuesto']) / 100
        )
        axes[1].barh(list(y_pos), hits_adme['Score Compuesto'].values,
                    color=colores_bar, alpha=0.9, edgecolor='white')
        axes[1].set_yticks(list(y_pos))
        axes[1].set_yticklabels([c.split()[0][:15] for c in hits_adme['Compuesto']],
                               fontsize=9)
        axes[1].set_xlabel('Score Compuesto (0-100)', fontsize=11)
        axes[1].set_title('Ranking de Candidatos\nScore Multicriterio', 
                        fontsize=12, fontweight='bold')
        axes[1].grid(True, alpha=0.3, axis='x')
        
        plt.suptitle('Análisis Multicriterio de Hits — Mpro SARS-CoV-2', 
                    fontsize=13, fontweight='bold')
        plt.tight_layout()
        plt.show()
    
    return hits_adme

if RDKIT_OK and len(df_resultados) > 0 and len(df) > 0:
    df_hits = priorizar_hits(df_resultados, df)
else:
    print("Necesitas ejecutar las celdas anteriores primero.")

## 7. Ejercicios Prácticos

### Ejercicio 1 (Básico)
Usando la función `calcular_descriptores`, analiza los siguientes compuestos y determina cuáles pasan el filtro de Lipinski:
- Atorvastatina: `O=C(O)C[C@@H](O)C[C@@H](O)CCc1c(-c2ccccc2F)c(-c2ccccc2)n(C(C)C)c1C(=O)Nc1ccccc1`
- Paclitaxel: `O=C(NC1C2OC2CC1OC(=O)c1ccccc1)c1ccccc1`
- Vitamina B12 (cyanocobalamin) — busca su SMILES en PubChem

### Ejercicio 2 (Intermedio)
Descarga los primeros 50 compuestos con IC₅₀ < 1 μM para la **SARS-CoV-2 Mpro** de la base de datos [ChEMBL](https://www.ebi.ac.uk/chembl/):
1. Aplica los filtros ADME
2. Ejecuta la función de cribado simulado con estos compuestos
3. ¿Qué porcentaje pasa todos los filtros?

### Ejercicio 3 (Avanzado)
Diseña un **flujo de cribado completo** para encontrar inhibidores de la **acetilcolinesterasa (AChE)** (PDB: 4EY7):
1. Construye una biblioteca de 20 compuestos entre alcaloides naturales y sus derivados sintéticos
2. Aplica filtros Lipinski y PAINS
3. Calcula las métricas de enriquecimiento si tienes datos de actividad
4. Justifica cuáles 3 compuestos seleccionarías para síntesis y evaluación experimental

In [ ]:
# Ejercicio 1: Analizar atorvastatina
atorvastatina_smiles = "O=C(O)C[C@@H](O)C[C@@H](O)CCc1c(-c2ccccc2F)c(-c2ccccc2)n(C(C)C)c1C(=O)Nc1ccccc1"
d = calcular_descriptores("Atorvastatina", atorvastatina_smiles)
if d:
    print(f"Atorvastatina:")
    for k, v in d.items():
        if k != 'Pasa Filtros':
            print(f"  {k}: {v}")
    print(f"  → Pasa todos los filtros: {'✓' if d['Pasa Filtros'] else '✗'}")
# Tu código para los ejercicios 2 y 3 aquí...

## 8. Referencias

1. Shoichet, B.K. (2004). Virtual screening of chemical libraries. *Nature*, 432, 862-865.
2. Lipinski, C.A. et al. (1997). Experimental and computational approaches to estimate solubility and permeability in drug discovery. *Adv. Drug Del. Rev.*, 23, 3-25.
3. Jin, Z. et al. (2020). Structure of Mpro from SARS-CoV-2 and discovery of its inhibitors. *Nature*, 582, 289-293.
4. Baell, J.B. & Holloway, G.A. (2010). New substructure filters for removal of pan assay interference compounds (PAINS) from screening libraries. *J. Med. Chem.*, 53, 2719-2740.
5. Trott, O. & Olson, A.J. (2010). AutoDock Vina: Improving the speed and accuracy of docking. *J. Comput. Chem.*, 31, 455-461.

---

## 📚 Recursos Adicionales

### Bases de Datos de Compuestos
- [ZINC](https://zinc15.docking.org/) — Millones de compuestos para cribado virtual
- [ChEMBL](https://www.ebi.ac.uk/chembl/) — Datos de actividad biológica
- [PubChem](https://pubchem.ncbi.nlm.nih.gov/) — Base de datos química del NCBI
- [DrugBank](https://www.drugbank.ca/) — Fármacos y candidatos

### Servidores de Cribado Virtual
- [DockBlaster](http://blaster.docking.org/) — Cribado virtual con DOCK
- [SwissDock](http://www.swissdock.ch/) — Docking online gratuito
- [AutoDock Vina](https://vina.scripps.edu/) — Software libre de docking

### Evaluación
- [DUD-E](http://dude.docking.org/) — Benchmarking de métodos de cribado virtual

---

## ✅ Verificación de Aprendizaje

Al finalizar esta actividad deberías ser capaz de:

- ✅ Explicar el flujo de trabajo de un cribado virtual por docking
- ✅ Aplicar filtros de Lipinski y Veber a una biblioteca de compuestos
- ✅ Identificar y eliminar compuestos PAINS
- ✅ Ejecutar un cribado automatizado con AutoDock Vina
- ✅ Calcular curvas ROC y factores de enriquecimiento
- ✅ Priorizar hits usando criterios multicriterio

---

<div align="center">

## 🎉 ¡Felicitaciones!

Has completado la **Actividad 4.8: Cribado Virtual**  
¡Y con ella, el **Módulo 4 completo**! 🏆

[![Anterior](https://img.shields.io/badge/⬅️_Actividad_4.7-Docking_Proteína_Proteína-blue.svg)](07_docking_proteina_proteina.ipynb)
[![Siguiente módulo](https://img.shields.io/badge/Módulo_5_➡️-Dinámica_Molecular-green.svg)](../modulo_05_dinamica_molecular/)

---

📚 **[Volver al Módulo 4](README.md)** | 🏠 **[Inicio del Curso](../README.md)**

---

**Universidad de Caldas - Departamento de Química**  
*Química Computacional 173G7G*

</div>